In [1]:
import sqlite3

# Connect to database
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

# 🔹 Create tables

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    city TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_name TEXT,
    enrollment_date TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS grades (
    grade_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_name TEXT,
    score INTEGER
)
""")

# 🔹 Insert data

students_data = [
    (1, "Alice", "alice@email.com", "Mumbai"),
    (2, "Bob", "bob@email.com", "Delhi"),
    (3, "Carol", "carol@email.com", "Bangalore"),
    (4, "David", "david@email.com", "Chennai"),
    (5, "Emma", "emma@email.com", "Pune")
]

enrollments_data = [
    (101, 1, "Python", "2024-01-15"),
    (102, 1, "SQL", "2024-02-01"),
    (103, 2, "Data Science", "2024-01-20"),
    (104, 3, "Python", "2024-02-10"),
    (105, 5, "Machine Learning", "2024-01-25")
]

grades_data = [
    (201, 1, "Python", 85),
    (202, 1, "SQL", 90),
    (203, 2, "Data Science", 78),
    (204, 4, "Web Development", 88)
]

# 🔹 Insert into tables
cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students_data)
cursor.executemany("INSERT INTO enrollments VALUES (?, ?, ?, ?)", enrollments_data)
cursor.executemany("INSERT INTO grades VALUES (?, ?, ?, ?)", grades_data)

# Commit and close
conn.commit()
conn.close()

print("Tables created and data inserted successfully!")

Tables created and data inserted successfully!


In [3]:
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

In [12]:
query="SELECT s.name, e.course_name \
FROM students AS s \
INNER JOIN enrollments AS e \
ON s.student_id = e.student_id"

cursor.execute(query)
print("-------------------QUERY-1------------------")
print(cursor.fetchall())

query="SELECT s.name, e.course_name \
FROM students AS s \
LEFT JOIN enrollments AS e \
ON s.student_id = e.student_id"

cursor.execute(query)

print("-------------------QUERY-2------------------")
print(cursor.fetchall())


-------------------QUERY-1------------------
[('Alice', 'Python'), ('Alice', 'SQL'), ('Bob', 'Data Science'), ('Carol', 'Python'), ('Emma', 'Machine Learning')]
-------------------QUERY-2------------------
[('Alice', 'Python'), ('Alice', 'SQL'), ('Bob', 'Data Science'), ('Carol', 'Python'), ('David', None), ('Emma', 'Machine Learning')]


## Explain the difference from INNER JOIN and which student appears with NULL course.

The left join will return all the rows from the left table and the matching rows from the right table. Now, the column from right table will have null values to unmatching rows with the left table. The student 'David' will have null value in cousename column.




In [13]:
#Thisis without usig alias. Without using allias we, might have to givethe table name everytime. 
#If the table names are large, then we have to repeate the tablename repeatedly.
query="SELECT students.name, enrollments.course_name \
FROM students  \
INNER JOIN enrollments \
ON students.student_id = enrollments.student_id"

cursor.execute(query)
print("-------------------QUERY-1------------------")
print(cursor.fetchall())

-------------------QUERY-1------------------
[('Alice', 'Python'), ('Alice', 'SQL'), ('Bob', 'Data Science'), ('Carol', 'Python'), ('Emma', 'Machine Learning')]


## Task2: Multiple Table Joins

In [14]:
query = "SELECT s.name, e.course_name, g.score FROM students AS s LEFT JOIN enrollments AS e ON s.student_id = e.student_id \
LEFT JOIN grades AS g ON s.student_id = g.student_id AND e.course_name = g.course_name"

cursor.execute(query)
print("-------------------QUERY-3------------------")
print(cursor.fetchall())


-------------------QUERY-3------------------
[('Alice', 'Python', 85), ('Alice', 'SQL', 90), ('Bob', 'Data Science', 78), ('Carol', 'Python', None), ('David', None, None), ('Emma', 'Machine Learning', None)]


### Explain:

1. Why we use LEFT JOIN instead of INNER JOIN here - We usedleft join because, er need all the rows form theleft table and mapping ones from th right table
2. What happens to students with no enrollments - Will have null values.
3. What happens to enrollments with no grades - Will have null values
4. Why the join condition includes both student_id AND course_name - The grades table is joining with two tables, so need to povide the mapping columns for both the tables.

Answer the following conceptual questions:

1. Why normalize data? Explain why student information is split across three tables instead of one large table with all data repeated. - This will have all redundant data all also will lead to a large table, which will in turn lead to efficiency issues while selecting data from table.

2. When to use each join type:

      1. Use INNER JOIN when: We need exactly the matching rows of all the tables.
      2. Use LEFT JOIN when: When you need all the rows from left table and only matching rows from right table with left table.


In [21]:
import pandas as pd

students_df = pd.read_sql("select * from students",conn)
enrollments_df = pd.read_sql("select * from enrollments",conn)
grades_df=pd.read_sql("select * from grades",conn)


# First merge students with enrollments
result = pd.merge(students_df, enrollments_df, 
                  on='student_id', how='left')

# Then merge with grades
result = pd.merge(result, grades_df, 
                  on=['student_id', 'course_name'], how='left')
result

,student_id,name,email,city,enrollment_id,course_name,enrollment_date,grade_id,score
0,1,Alice,alice@email.com,Mumbai,101.0,Python,2024-01-15,201.0,85.0
1,1,Alice,alice@email.com,Mumbai,102.0,SQL,2024-02-01,202.0,90.0
2,2,Bob,bob@email.com,Delhi,103.0,Data Science,2024-01-20,203.0,78.0
3,3,Carol,carol@email.com,Bangalore,104.0,Python,2024-02-10,NaN,NaN
4,4,David,david@email.com,Chennai,NaN,NaN,NaN,NaN,NaN
5,5,Emma,emma@email.com,Pune,105.0,Machine Learning,2024-01-25,NaN,NaN
